# ETL - Etapa 2: Tratamento
- Lendo a base bruta, padroniza nomes e tipos, trata nulos e duplicados.

- Entrada: data/raw/cotacoes_brutas.csv

- Saída: data/processed/cotacoes_limpas.csv

## Bibliotecas Leitura e inicio dos dados brutos (Obs)

In [201]:
# Bibliotecas:
from pathlib import Path
import pandas as pd

# Pasta RAIZ do projeto:
raiz = Path.cwd().parent

# Local do arquivo CSV com os dados brutos:
caminho = raiz / "data" / "raw" / "cotacoes_brutas.csv"

# Leitura dos dados brutos:
df_raw = pd.read_csv(caminho, sep=",", decimal=".", encoding="utf-8")

# Local de armazenamento da base tratada:
pasta_processed = raiz / "data" / "processed"

df_raw.head()  # Visualização do inicio dos dados brutos

,Date,Adj Close,Close,High,Low,Open,Volume,nome_acao
0,2021-01-04,32.996513,37.310001,38.195000,37.250000,37.924999,7219800,WEGE3
1,2021-01-05,35.021744,39.599998,39.599998,37.000000,37.325001,16633600,WEGE3
2,2021-01-06,35.950352,40.650002,41.735001,39.064999,39.685001,21507400,WEGE3
3,2021-01-07,37.436138,42.330002,42.505001,40.919998,41.189999,15129600,WEGE3
4,2021-01-08,39.700172,44.889999,44.994999,42.424999,42.474998,17381000,WEGE3


**Obeservação:**
- Nas colunas Close, High, Low e Open, alguns apresentaram valores excesso de casas decimais. 
    - Irei corrigir para apenas duas casas decimais.

## Informação Geral dos e descrições dos dados brutos (Obs)

In [202]:
# Informações dos sobres os dados
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 8940 entries, 0 to 8939
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Date       8940 non-null   str    
 1   Adj Close  8940 non-null   float64
 2   Close      8940 non-null   float64
 3   High       8940 non-null   float64
 4   Low        8940 non-null   float64
 5   Open       8940 non-null   float64
 6   Volume     8940 non-null   int64  
 7   nome_acao  8940 non-null   str    
dtypes: float64(5), int64(1), str(2)
memory usage: 690.6 KB


**Observação:**
- Total de linhas: 8940, índices de 0 ao 8939.
- Total de Colunas: 8.
- Apenas uma coluna com nome em português. `nome_acao`
- A coluna `Date` não foi lida como datetime64.
- Nenhum valor nulo encontrado.

Descrições das colunas:
| Coluna| Tipo lido pelo pandas | Descrição |
|---|---|---|
| Date | `str` | Data da cotação. |
| Adj Close | `float` | Preço de fechamento ajustado por dividendos e desdobramentos. |
| Close | `float` | Preço de fechamento. |
| High | `float` | Preço máximo. |
| Low | `float` | Preço mínimo. |
| Open | `float` | Preço de abertura. |
| Volume | `int` | Quantidade de ações negociadas do dia. |
| nome_acao | `str` | Nome do ativo. |



# Tratamento dos dados brutos

## Renomeando as colunas

In [203]:
# Renomeando as colunas para português
df = df_raw.rename(
    columns={
        "Date": "data",
        "Adj Close": "fechamento_ajustado",
        "Close": "fechamento",
        "High": "maxima",
        "Low": "minima",
        "Open": "abertura",
        "Volume": "volume",
    }
)

## Corrigindo os decimais

In [204]:
# Corrindo as casas decimais para apenas 2 casas:
# A coluna fechamento_ajustado não irei corrigir, pois é um valor calculado.
lista_decimais = [
    "fechamento",
    "maxima",
    "minima",
    "abertura",
]

# para cada coluna, corrigir casas decimais:
for decimal in lista_decimais:
    df[decimal] = df[decimal].round(2)

df.head()

,data,fechamento_ajustado,fechamento,maxima,minima,abertura,volume,nome_acao
0,2021-01-04,32.996513,37.31,38.19,37.25,37.92,7219800,WEGE3
1,2021-01-05,35.021744,39.60,39.60,37.00,37.33,16633600,WEGE3
2,2021-01-06,35.950352,40.65,41.74,39.06,39.69,21507400,WEGE3
3,2021-01-07,37.436138,42.33,42.51,40.92,41.19,15129600,WEGE3
4,2021-01-08,39.700172,44.89,44.99,42.42,42.47,17381000,WEGE3


## Corrigindo as datas

In [205]:
# Corrigindo a leitura da coluna (data) para (datetime64)
df["data"] = pd.to_datetime(df["data"])

print(type(df["data"][0]))  # Imprime o tipo de dados da coluna

<class 'pandas.Timestamp'>


## Verificando a existencia de duplicados

In [206]:
# Verifica se a mesma ação aparece mais de uma vez na mesma data
duplicados = df.duplicated(subset=["data", "nome_acao"]).sum()
print(f"Linhas duplicadas: {duplicados}")

Linhas duplicadas: 0


## Informações estatísticas (Obs)

In [207]:
# Informações descritivas
df.describe()

,data,fechamento_ajustado,fechamento,maxima,minima,abertura,volume
count,8940,8940.000000,8940.000000,8940.000000,8940.000000,8940.000000,8.940000e+03
mean,2022-07-02 03:38:24.966443,24.628770,31.794547,32.270322,31.339704,31.814386,2.320298e+07
min,2021-01-04 00:00:00,5.697405,8.370000,8.680000,8.210000,8.340000,0.000000e+00
25%,2021-10-01 00:00:00,14.077126,19.240000,19.530000,18.970000,19.267500,6.604489e+06
50%,2022-07-04 00:00:00,18.641513,24.660000,25.060000,24.290000,24.660000,1.463085e+07
75%,2023-03-30 00:00:00,31.308897,34.650000,35.140000,34.210000,34.640000,3.153068e+07
max,2023-12-28 00:00:00,71.032074,118.720000,120.450000,115.250000,119.800000,4.991948e+08
std,NaN,14.839196,20.056202,20.320034,19.794548,20.075731,2.559222e+07


**Observação:**
- Primeira data registrada e menor: 2021/01/04, pois nos dias 01 foi feriado e 2 a 03/01/2021 foi um final de semana.
- Ultima data registrada e maior: 2023/12/28, pois nos dias 29 a 31/12/2023 não houve pregão.
- A coluna `volume` dentro do `describe()` aprestou um valor igual a zero (0.0)

## Verificando a existência de: (Obs)
- Valores negativos
- Maxima menor que Minima

In [208]:
# Verificando a existência de valores negativos e quais são zero
print(f"Valores <= 0 presentes em cada coluna:\n{(df[[
        "fechamento_ajustado",
        "fechamento",
        "maxima",
        "minima",
        "abertura",
        "volume"
    ]] <= 0).sum().to_string()}")

# Valore máximo do dia não pode ser menor que o mínimo
print(f"\nMáxima <  Mínima: {(df["maxima"] < df["minima"]).sum()}\n")

# Percentual em relação ao total de linhas do dataframe
quantidade = len(df[df["volume"] == 0])
percentual = (quantidade / len(df) * 100)
print(f"Percentual dessa linhas: {percentual:.2f}%")

# Quais linhas tem o volume igual a zero?
print(f"\nQuem são as linhas com volume igual a zero:")
display(df[df["volume"] == 0])


Valores <= 0 presentes em cada coluna:
fechamento_ajustado    0
fechamento             0
maxima                 0
minima                 0
abertura               0
volume                 3

Máxima <  Mínima: 0

Percentual dessa linhas: 0.03%

Quem são as linhas com volume igual a zero:


,data,fechamento_ajustado,fechamento,maxima,minima,abertura,volume,nome_acao
431,2022-09-26,27.793694,30.57,30.57,30.57,30.57,0,WEGE3
4901,2022-09-26,28.074907,28.12,28.12,28.12,28.12,0,PRIO3
7306,2023-06-02,11.721806,15.78,15.78,15.78,15.78,0,BBDC4


**Observação:**
- 3 registros (WEGE3 e PRIO3 em 26/09/2022, BBDC4 em 02/06/2023)
    - apresentam volume zero e preços de abertura, máxima, mínima e fechamento idênticos, um indicativo de um erro de registro, pois parece que não foi negociado nesse dia, o que seria improvavel.
- Irei manter eles pois não iram impactar na análise, já que o percentual é de 0.03% (3 de 8940).

# Organização e Exportação da base limpa

In [209]:
# Ordenando por ação e data (para garantir a sequencia correta para os cálculos de retorno, médias móveis e drawdown) e refaz o indice.
df = df.sort_values(["nome_acao", "data"]).reset_index(drop=True)

# Salvando o arquivo tratado.
pasta_processed.mkdir(parents=True, exist_ok=True)
df.to_csv(pasta_processed / "catacoes_limpas.csv", index=False)

print(f"Base limpa salva em:\n{pasta_processed / "catacoes_limpas.csv"}")
print(f"{len(df)} linhas | {df["nome_acao"].nunique()} ativos")

Base limpa salva em:
c:\Users\ferna\Desktop\SCTEC-GERAL\SCTEC- Pratica - Mod_02\Semana_13\Projeto_Final\M2_Projeto_Final_Analise_B3\data\processed\catacoes_limpas.csv
8940 linhas | 12 ativos
